# Test pipeline trên trận mới: video → luồng YES → luồng NO (Colab, v5.1)

Chạy **hệ thống đã train** trên trận đấu mới (không có nhãn, không train gì):

| Bước | Việc làm |
|---|---|
| 1–4 | GPU, cấu hình, mount Drive, lấy code |
| 5 | Video 2 hiệp → npy TimeSformer 768 chiều |
| 6 | Phép ép 768 → **npy 8576 chiều** (phép ép đã học ở notebook train) |
| 7 | **Luồng YES**: model đã train bắt sự kiện + sinh câu bình luận → `results_dense_captioning.json` |
| 7b | Tuỳ chọn: **recall** của luồng YES so với sự kiện FIFA, với NMS 0.4 / 0.5 / 0.6 / 0.7 |
| 8 | **Luồng NO**: tìm khoảng trống giữa các sự kiện YES, LLM ElevenLabs nói thông tin từ metadata FIFA |
| 9 | Xem timeline YES + NO (v1) |
| **10** | **Luồng NO v2** (kịch bản 2.0 + Gemini): câu bắt buộc (chào, tỉ số hiệp 1, hết trận...) + thông tin nền |
| **11** | **Giọng đọc (Gemini TTS)**: 2 giọng (sự kiện / dẫn chương trình), ghép vào video từng hiệp |

Cell 8–9 (ElevenLabs, v1) vẫn giữ. Muốn dùng bản mới thì chạy cell 10–11 thay cho 8–9.

**Cần có trên Drive** (`MyDrive/soccer_project/`), **để Private**:
- `video_test/<giải>/<mùa>/<trận>/1_224p.mp4` và `2_224p.mp4`: hai hiệp đã cắt bằng `cut_halves.py`.
- `video_test/metadata/<trận>/`: kết quả `fifa_crawl.py` (`facts.csv`, `events.csv`, `match_info.csv`...). Tên thư mục trùng tên thư mục trận.
- Từ notebook train: `sn-models/<MODEL_NAME>/` (checkpoint `spotting/` và `caption/`), file phép ép trong `sn-models/`, weights TimeSformer `*.pdparams`.

**API key ElevenLabs**: đặt trong Colab Secrets (biểu tượng 🔑 bên trái) tên `ELEVENLABS_API_KEY`, bật quyền cho notebook. Không dán key vào cell. Không có key thì bước 8 chạy chế độ `template` (câu fact gốc, không LLM).

**API key Gemini** (cell 10–11): Colab Secrets, tên `GEMINI_API_KEY`, bật quyền cho notebook. Không dán key vào cell.

Chọn GPU: `Runtime → Change runtime type → T4 GPU`.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Cấu hình

- `MODEL_NAME`, `MAPPER_FILE`: model và phép ép **đi theo cặp**, lấy đúng cặp đã train (notebook train cell 2).
- `NMS_THRESHOLD`, `TOP_K`: chỉ dùng lúc suy luận, đổi không cần train lại. NMS thấp hơn thì báo nhiều sự kiện hơn (ít bỏ sót, dễ báo sai hơn).
- `NO_FLOW_GAMES`: chỉ chạy luồng NO (gọi LLM, tốn credit ElevenLabs) cho các trận này. `[]` = mọi trận.

In [ ]:
GITHUB_REPO = "https://github.com/ThuLinh3009/soccer_project.git"
GIT_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/soccer_project"
WORK_DIR = "/content/soccernet-caption-deltalab"   # code
TEST_DATA = "/content/test_data"                   # npy của các trận test
TEST_DIR_NAME = "video_test"                       # thư mục trên Drive chứa video + metadata trận test

MODEL_NAME = "gpt2-video60-mlp"                    # model đã train (sn-models/<MODEL_NAME>)
MAPPER_FILE = "mapper_timesformer_mlp_c4.pt"       # phép ép đã học cùng model đó
FEATURES = "baidu_from_timesformer.npy"
GPT_TYPE = "gpt2"
NMS_THRESHOLD = 0.4                                # test: recall 0,78, precision 0,75, ~110 sự kiện/trận
TOP_K = 3
TS_BATCH = 32

LLM = "elevenlabs"                                 # "template": không gọi LLM, dùng câu fact gốc
LLM_MODEL = "gemini-2.5-flash"
NO_FLOW_GAMES = ["2026-07-05 - Brazil 1 - 2 Norway"]   # trận chạy luồng NO (tiết kiệm credit); [] = tất cả

# v2 (cell 10–11): Gemini
# danh sách model thử lần lượt: model trước hết quota (gói miễn phí) thì dùng model sau
GEMINI_LLM_MODEL = "gemini-2.5-flash-lite,gemini-3.1-flash-lite,gemini-3.5-flash-lite"   # "template": không gọi LLM
GEMINI_TTS_MODEL = "gemini-3.8-flash-lite-tts,gemini-2.5-flash-preview-tts,gemini-3.1-flash-tts-preview"
VOICE_YES, VOICE_NO = "Puck", "Charon"             # giọng sự kiện / giọng dẫn chương trình
TTS_MAX_LINES = 30                                 # thử trước 30 câu đầu (tiết kiệm quota). 0 = cả trận
WORDS_PER_SECOND = 2.6     # tốc độ đọc dùng để tính số từ tối đa của mỗi câu; cell 11 in ra tốc độ đo được
TTS_MAX_SPEED = 1.25       # câu đọc dài hơn khoảng trống thì tăng tốc tối đa 1,25 lần (1 = không tăng tốc)
TTS_TIMELINE = "v2"        # "v2": timeline cell 10 | "v1": timeline cell 8 (output/timeline, no_flow_llm.py)

## 3. Mount Google Drive, tìm trận test, model và weights

In [ ]:
from google.colab import drive
import os, glob
drive.mount("/content/drive", force_remount=True)

if not os.path.isdir(DRIVE_ROOT):
    found = [p for p in glob.glob("/content/drive/MyDrive/**/soccer_project", recursive=True)
             + glob.glob("/content/drive/Shareddrives/*/soccer_project") if os.path.isdir(p)]
    assert found, f"Không thấy {DRIVE_ROOT}: sửa DRIVE_ROOT ở cell 2"
    DRIVE_ROOT = found[0]

TEST_SRC = f"{DRIVE_ROOT}/{TEST_DIR_NAME}"
DRIVE_MODEL_DIR = f"{DRIVE_ROOT}/sn-models"
MAPPER = f"{DRIVE_MODEL_DIR}/{MAPPER_FILE}"
FEAT_CACHE = f"{TEST_SRC}/features"                # npy 768 chiều lưu lại trên Drive
OUT_YES = f"{TEST_SRC}/output/yes"
OUT_TIMELINE = f"{TEST_SRC}/output/timeline"
weights = sorted(glob.glob(f"{DRIVE_ROOT}/**/*.pdparams", recursive=True))
TS_WEIGHTS = weights[0] if weights else None

videos = sorted(glob.glob(f"{TEST_SRC}/*/*/*/[12]_224p.mp4"))
GAMES = sorted({os.path.relpath(os.path.dirname(v), TEST_SRC) for v in videos})
print("DRIVE_ROOT:", DRIVE_ROOT)
for g in GAMES:
    halves = sorted(os.path.basename(v)[0] for v in videos if os.path.dirname(v).endswith(g))
    meta = f"{TEST_SRC}/metadata/{os.path.basename(g)}/facts.csv"
    print(f"{'✅' if halves == ['1', '2'] else '⚠️ thiếu hiệp'} hiệp {halves} | metadata {'✅' if os.path.exists(meta) else '❌'} | {g}")
print("model   :", MODEL_NAME, "✅" if os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/caption/model.pth.tar") else "❌ không thấy checkpoint")
print("phép ép :", MAPPER, "✅" if os.path.exists(MAPPER) else "❌")
print("weights :", TS_WEIGHTS or "❌ KHÔNG THẤY file .pdparams")
assert GAMES and os.path.exists(MAPPER) and TS_WEIGHTS
assert os.path.exists(f"{DRIVE_MODEL_DIR}/{MODEL_NAME}/spotting/model.pth.tar")

## 4. Lấy code từ GitHub và cài thư viện

In [ ]:
if os.path.isdir(f"{WORK_DIR}/.git"):
    !git -C {WORK_DIR} fetch -q origin && git -C {WORK_DIR} reset -q --hard origin/{GIT_BRANCH}
else:
    !git clone -q -b {GIT_BRANCH} {GITHUB_REPO} {WORK_DIR}
%cd {WORK_DIR}
!git log --oneline -1
!grep -m1 "^## v" CHANGELOG.md
!pip install -q SoccerNet av tiktoken transformers wandb pycocoevalcap psutil six websockets
!pip install -q --no-deps git+https://github.com/Maluuba/nlg-eval.git@master

import subprocess
os.environ["WANDB_MODE"] = "offline"

def run(cmd):
    # chạy lệnh, in log trực tiếp; báo lỗi ngay tại cell này nếu lệnh thất bại
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=WORK_DIR)
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:
            continue
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}): xem log ngay phía trên")

# models/ trỏ thẳng tới sn-models trên Drive (checkpoint đã train)
if not os.path.islink("models"):
    !rm -rf models
!ln -sfn "{DRIVE_MODEL_DIR}" models

## 5. Video → npy TimeSformer 768 chiều

Mỗi hiệp mất khoảng 5–10 phút trên T4. Kết quả lưu lại trên Drive (`video_test/features/`), chạy lại thì không trích lại.

In [ ]:
import shutil
def sync(src_root, dst_root):
    for f in glob.glob(f"{src_root}/**/*_timesformer768.npy", recursive=True):
        dst = os.path.join(dst_root, os.path.relpath(f, src_root))
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copyfile(f, dst)

if os.path.isdir(FEAT_CACHE):
    sync(FEAT_CACHE, TEST_DATA)
try:
    run(["python", "timesformer_extract.py", "--weights", TS_WEIGHTS, "--video_root", TEST_SRC,
         "--video_name", "{half}_224p.mp4", "--out_root", TEST_DATA, "--batch_size", str(TS_BATCH), "--games", *GAMES])
finally:
    sync(TEST_DATA, FEAT_CACHE)
missing = [f"{g}/{h}" for g in GAMES for h in (1, 2) if not os.path.exists(f"{TEST_DATA}/{g}/{h}_timesformer768.npy")]
assert not missing, f"Còn thiếu: {missing}. Chạy lại cell này"

## 6. Phép ép 768 → npy 8576 chiều

Dùng **đúng phép ép đã học** ở notebook train (không học lại). Mỗi hiệp ra `1_/2_baidu_from_timesformer.npy`, kích thước (số giây, 8576).

In [ ]:
run(["python", "feature_mapper.py", "apply", "--soccernet_path", TEST_DATA, "--mapper", MAPPER,
     "--games", *GAMES, "--out_name", FEATURES, "--overwrite"])
import numpy as np
for g in GAMES:
    print([np.load(f"{TEST_DATA}/{g}/{h}_{FEATURES}", mmap_mode="r").shape for h in (1, 2)], g)

## 7. Luồng YES: bắt sự kiện + sinh câu bình luận

Model đã train (`spotting` + `caption`) chạy trên npy 8576 chiều của trận mới: tìm thời điểm sự kiện (NMS), rồi GPT-2 sinh một câu cho mỗi sự kiện. Không có nhãn nên không chấm điểm.
Kết quả: `video_test/output/yes/<trận>/results_dense_captioning.json`.

In [ ]:
run(["python", "infer_match.py", "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE,
     "--nms_threshold", str(NMS_THRESHOLD), "--top_k", str(TOP_K), "--out_dir", OUT_YES,
     "--game_dirs", *[f"{TEST_DATA}/{g}" for g in GAMES]])

## 7b. Recall của luồng YES so với sự kiện FIFA (tuỳ chọn)

Trận mới không có nhãn SoccerNet nên so với `events.csv` của FIFA: bàn thắng, dứt điểm, phạt góc, phạm lỗi, việt vị, thẻ, thay người.
FIFA chỉ ghi tới **phút**, nên một sự kiện được tính là "bắt được" nếu có sự kiện YES trong phút đó (±15 giây).

- `recall`: tỉ lệ sự kiện FIFA bắt được. Đây là con số chính.
- `FIFA overlap`: tỉ lệ sự kiện YES nằm gần một sự kiện FIFA. FIFA không ghi mọi pha bóng (ném biên, phát bóng...) nên đây **không phải** precision, chỉ là mức thấp nhất.
- `random`: kết quả nếu đặt **cùng số sự kiện ở thời điểm ngẫu nhiên**. Cửa sổ khớp rộng (~90 giây) nên ngẫu nhiên cũng đạt recall khá cao: model chỉ tốt khi **vượt xa** mức ngẫu nhiên.

Chỉ chạy bắt sự kiện (không sinh câu) cho từng ngưỡng trong `NMS_LIST`, lưu vào `output/yes_nms<ngưỡng>/`, không ghi đè `output/yes`.

In [ ]:
import json, pandas as pd
NMS_LIST = [0.4, 0.5, 0.6, 0.7]
rows = []
for nms in NMS_LIST:
    out = f"{TEST_SRC}/output/yes_nms{nms}"
    run(["python", "infer_match.py", "--model_name", MODEL_NAME, "--features", FEATURES, "--gpt_type", GPT_TYPE,
         "--nms_threshold", str(nms), "--out_dir", out, "--no_caption",
         "--game_dirs", *[f"{TEST_DATA}/{g}" for g in GAMES]])
    for g in GAMES:
        name = os.path.basename(g)
        events = f"{TEST_SRC}/metadata/{name}/events.csv"
        if not os.path.exists(events):
            print("⚠️ không có", events); continue
        res = subprocess.run(["python", "eval_vs_fifa.py", "--yes_dir", f"{out}/{name}", "--events", events],
                             capture_output=True, text=True, cwd=WORK_DIR)
        print(res.stdout.strip() or res.stderr[-2000:])
        line = [l for l in res.stdout.splitlines() if l.startswith("RESULT ")]
        if line:
            rows.append({"NMS": nms, "game": name[:35], **json.loads(line[0][7:])})
table = pd.DataFrame(rows)
display(table.round(3))
table.to_csv(f"{TEST_SRC}/output/recall_vs_fifa.csv", index=False)

## 8. Luồng NO: LLM ElevenLabs trong các khoảng trống

- Khoảng trống = khoảng giữa hai sự kiện **của luồng YES ở bước 7**.
- Mỗi khoảng trống chọn một fact trong `facts.csv` (metadata FIFA), LLM viết lại thành câu nói. Câu có con số không có trong ngữ cảnh thì dùng câu fact gốc.
- Không train, không cần nhãn.

In [ ]:
from google.colab import userdata
llm = LLM
try:
    os.environ["ELEVENLABS_API_KEY"] = userdata.get("ELEVENLABS_API_KEY")
except Exception as e:
    print("⚠️ không đọc được secret ELEVENLABS_API_KEY:", type(e).__name__, "→ chạy chế độ template")
    llm = "template"

if llm == "elevenlabs":
    # kiểm tra key trước: 401 = key sai / đã xoá / thiếu quyền ElevenAgents
    import requests
    key = os.environ["ELEVENLABS_API_KEY"].strip().strip("\"'").strip()
    os.environ["ELEVENLABS_API_KEY"] = key
    r = requests.get("https://api.elevenlabs.io/v1/convai/agents", params={"page_size": 1},
                     headers={"xi-api-key": key}, timeout=30)
    print(f"kiểm tra key ({len(key)} ký tự, bắt đầu '{key[:3]}'):", r.status_code, "OK" if r.ok else r.text[:300])
    assert r.ok, ("ElevenLabs không nhận key: tạo key mới (bật quyền ElevenAgents), dán lại vào secret "
                  "ELEVENLABS_API_KEY, rồi chạy lại cell này")

for g in GAMES:
    name = os.path.basename(g)
    if NO_FLOW_GAMES and name not in NO_FLOW_GAMES:
        print("bỏ qua (không có trong NO_FLOW_GAMES):", name)
        continue
    run(["python", "no_flow_llm.py", "--metadata", f"{TEST_SRC}/metadata/{name}", "--video_dir", f"{TEST_SRC}/{g}",
         "--yes_events", f"{OUT_YES}/{name}/results_dense_captioning.json", "--llm", llm, "--llm_model", LLM_MODEL,
         "--agent_cache", f"{DRIVE_MODEL_DIR}/elevenlabs_agent.json", "--out_dir", OUT_TIMELINE])

## 9. Timeline YES + NO

`[EVENT]`: câu của luồng YES (model). `[INFO ]`: câu của luồng NO (LLM từ metadata).

In [ ]:
for g in GAMES:
    path = f"{OUT_TIMELINE}/{os.path.basename(g)}/commentary_timeline.txt"
    if not os.path.exists(path):
        print("chưa có timeline:", g)
        continue
    lines = open(path, encoding="utf-8").read().splitlines()
    print(f"\n===== {g}: {sum('[EVENT]' in l for l in lines)} câu YES, {sum('[INFO ]' in l for l in lines)} câu NO")
    print("\n".join(lines[:60]))

## 10. Luồng NO v2: kịch bản 2.0 + Gemini

- Theo `NO_FLOW_SCRIPT.md`: **câu bắt buộc** (chào mừng, 2 đội + HLV, ý nghĩa trận, tin nóng bàn thắng / thẻ / thay người, nhắc tỉ số mỗi ~10 phút, hết hiệp 1, đầu hiệp 2, phút 80', bù giờ, hết trận) và **thông tin nền** trong khoảng trống dài.
- Mỗi câu là khuôn có biến điền từ metadata FIFA (dùng cho mọi trận). Gemini (`GEMINI_LLM_MODEL`) nói lại tự nhiên; câu sai số, sai tên, quá dài thì dùng lại khuôn.
- Câu YES được bỏ `[PLAYER]` / `[TEAM]` ("the player", "the team") để đọc thành tiếng.
- Kết quả: `video_test/output/timeline_v2/<trận>/commentary_timeline.txt` và `.json`.

In [ ]:
from google.colab import userdata
llm_v2 = "template" if GEMINI_LLM_MODEL == "template" else "gemini"
try:
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY").strip().strip("\"'").strip()
except Exception as e:
    print("⚠️ không đọc được secret GEMINI_API_KEY:", type(e).__name__, "→ chạy chế độ template (không LLM, không TTS)")
    llm_v2 = "template"
if "GEMINI_API_KEY" in os.environ:
    run(["python", "gemini_client.py", "--check"])

OUT_TIMELINE_V2 = f"{TEST_SRC}/output/timeline_v2"
for g in GAMES:
    name = os.path.basename(g)
    if NO_FLOW_GAMES and name not in NO_FLOW_GAMES:
        print("bỏ qua (không có trong NO_FLOW_GAMES):", name)
        continue
    run(["python", "no_flow_v2.py", "--metadata", f"{TEST_SRC}/metadata/{name}", "--video_dir", f"{TEST_SRC}/{g}",
         "--yes_events", f"{OUT_YES}/{name}/results_dense_captioning.json", "--llm", llm_v2,
         "--llm_model", GEMINI_LLM_MODEL if llm_v2 == "gemini" else "none", "--out_dir", OUT_TIMELINE_V2,
         "--wps", str(WORDS_PER_SECOND)])
    path = f"{OUT_TIMELINE_V2}/{name}/commentary_timeline.txt"
    lines = open(path, encoding="utf-8").read().splitlines()
    no = [l for l in lines if "[EVENT]" not in l and "[MUTED]" not in l and "[FIFA ]" not in l]
    print(f"\n===== {name}: {len(lines) - len(no)} câu YES, {len(no)} câu NO. Các câu NO:")
    print("\n".join(no))

## 11. Giọng đọc: Gemini TTS, ghép vào video

- 2 giọng: `VOICE_YES` đọc câu sự kiện (luồng YES), `VOICE_NO` đọc câu dẫn chương trình (luồng NO). Bàn thắng đọc giọng hào hứng.
- Mỗi câu thành một file wav (lưu cache trên Drive: chạy lại không tốn quota cho câu đã đọc), đặt đúng thời điểm, ghép thành audio từng hiệp, rồi lồng vào video (tiếng gốc của video giảm còn 25%).
- Thời gian (v5.1) theo độ dài thật của từng clip:
  - câu NO nối tiếp (chuỗi A1→A2→A3, C, G, hoặc cách câu trước ≤ 5 s) bắt đầu ở **giây kế tiếp** sau khi câu trước đọc xong;
  - câu YES không bao giờ đọc trước sự kiện; câu trước chưa đọc xong thì lùi đến giây kế tiếp;
  - clip dài hơn khoảng trống đến câu sau thì tăng tốc (không đổi cao độ), tối đa `TTS_MAX_SPEED`.
- `audio_timeline.json`: `gameTime` → `gameTimeEnd` = lúc câu thực sự được nghe; `planned` = giờ trong kịch bản.
- `TTS_TIMELINE = "v1"`: đọc timeline của cell 8 (`output/timeline`), câu `[PLAYER] ([TEAM])` đọc thành "the player (the team)".
- `TTS_MAX_LINES = 30`: thử 30 câu đầu trước. Quota TTS miễn phí có giới hạn mỗi ngày; hết quota thì cell dừng, giữ các câu đã đọc, hôm sau chạy tiếp.
- Kết quả: `video_test/output/timeline_v2/<trận>/audio/1_commentary.mp4`, `2_commentary.mp4` (+ wav).

In [ ]:
for g in GAMES:
    name = os.path.basename(g)
    base = OUT_TIMELINE_V2 if TTS_TIMELINE == "v2" else OUT_TIMELINE
    timeline = f"{base}/{name}/commentary_timeline.json"
    if not os.path.exists(timeline):
        print(f"chưa có timeline {TTS_TIMELINE} (chạy cell {10 if TTS_TIMELINE == 'v2' else 8}):", name)
        continue
    run(["python", "tts_v2.py", "--timeline", timeline, "--video_dir", f"{TEST_SRC}/{g}",
         "--tts_model", GEMINI_TTS_MODEL, "--voice_yes", VOICE_YES, "--voice_no", VOICE_NO,
         "--max_lines", str(TTS_MAX_LINES), "--max_speed", str(TTS_MAX_SPEED)])

from IPython.display import Audio, display
for g in GAMES:
    base = OUT_TIMELINE_V2 if TTS_TIMELINE == "v2" else OUT_TIMELINE
    wav = f"{base}/{os.path.basename(g)}/audio/1_commentary.wav"
    if os.path.exists(wav):
        print("Nghe thử 3 phút đầu hiệp 1:", os.path.basename(g))
        import wave, numpy as np
        with wave.open(wav) as w:
            audio = np.frombuffer(w.readframes(w.getframerate() * 180), dtype=np.int16)
            display(Audio(audio, rate=w.getframerate()))